In [1]:
import torch
#use GPU if available 
#DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") #'cpu' # 'cuda' or 'cpu'
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(DEVICE)

mps


In [2]:
!git clone --single-branch --branch "develop" "https://github.com/gab-palmeri/aml-geolocalization.git"
!mv "aml-geolocalization/" "Team/"

Cloning into 'aml-geolocalization'...
remote: Enumerating objects: 225, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 225 (delta 65), reused 70 (delta 41), pack-reused 125
Receiving objects: 100% (225/225), 2.07 MiB | 4.36 MiB/s, done.
Resolving deltas: 100% (96/96), done.


In [3]:
torch.backends.cudnn.benchmark = True  # Provides a speedup

class dotdict(dict):
    """dot.notation access to dictionary attributes"""
    __getattr__ = dict.get
    __setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__

In [4]:
# CosPlace Groups parameters
COS_M = 10
ALPHA = 30
COS_N = 5
COS_L = 2
GROUPS_NUM = 1
MIN_IMAGES_PER_CLASS = 10
# Model parameters
BACKBONE = "cct384"   
# ["resnet18", "efficientnet_b2", "efficientnet_v2_s", "mobilenet_v3_small", "mobilenet_v3_large",
# ["convnext_tiny", "swin_tiny", "cct384"]
FC_OUTPUT_DIM = 512     # Output dimension of final fully connected layer
PRETRAIN = "imagenet"   # ["imagenet", "places", "gldv2"]
# Training parameters
AUGMENTATION_DEVICE = "mps"    # ["cuda", "cpu"]
USE_AMP_16 = AUGMENTATION_DEVICE == "cuda"       # use Automatic Mixed Precision
BATCH_SIZE = 32
EPOCHS_NUM = 3
ITERATIONS_PER_EPOCH = 10_000
LR = 0.00001                    # Learning rate  
CLASSIFIERS_LR = 0.01
# Data augmentation
BRIGHTNESS = 0.7
CONTRAST = 0.7
SATURATION = 0.7
HUE = 0.5
RANDOM_RESIZED_CROP = 0.5
# Validation / test parameters
INFER_BATCH_SIZE = 16           # Batch size for inference (validating and testing)
POSITIVE_DIST_THRESHOLD = 25    # distance in meters for a prediction to be considered a positive
# Resume parameters
RESUME_TRAIN = None     # path to checkpoint to resume, e.g. logs/.../last_checkpoint.pth
RESUME_MODEL = None     # Path to model to resume training from
# Other parameters
DEVICE = "mps"                     # ["cuda", "cpu"]
SEED = 0
NUM_WORKERS = 8
DATASET_FOLDER = "/content/small"   # path of the folder with train/val sets
SAVEDIR = BACKBONE + "_" + PRETRAIN


# dictionary for the parameters
args = {
    'M': COS_M, 'alpha': ALPHA, 'N': COS_N, 'L': COS_L, 'groups_num': GROUPS_NUM,
    'min_images_per_class': MIN_IMAGES_PER_CLASS, 'backbone': BACKBONE, "pretrain": PRETRAIN,
    'fc_output_dim': FC_OUTPUT_DIM, 'use_amp16': USE_AMP_16,
    'augmentation_device': AUGMENTATION_DEVICE, 'batch_size': BATCH_SIZE,
    'epochs_num': EPOCHS_NUM, 'iterations_per_epoch': ITERATIONS_PER_EPOCH,
    'lr': LR, 'classifiers_lr': CLASSIFIERS_LR, 'brightness': BRIGHTNESS,
    'contrast': CONTRAST, 'hue': HUE, 'saturation': SATURATION,
    'random_resized_crop': RANDOM_RESIZED_CROP, 'infer_batch_size': INFER_BATCH_SIZE,
    'positive_dist_threshold': POSITIVE_DIST_THRESHOLD, 'resume_train': RESUME_TRAIN,
    'resume_model': RESUME_MODEL, 'device': DEVICE, 'seed': SEED,
    'num_workers': NUM_WORKERS, 'dataset_folder': DATASET_FOLDER, 'save_dir': SAVEDIR,
    'val_set_folder': "", 'train_set_folder': "",
}

# this helps to reuse the code from the original CosPlace
args = dotdict(args)

In [7]:
import logging
import timm
from Team.model import network

backbone, feature_dim = network.get_backbone(args.backbone, args.pretrain)


Downloading: "https://shi-labs.com/projects/cct/checkpoints/finetuned/cct_14_7x2_384_imagenet.pth" to /Users/slotruglio/.cache/torch/hub/checkpoints/cct_14_7x2_384_imagenet.pth


  0%|          | 0.00/85.9M [00:00<?, ?B/s]

In [14]:
total_params = sum(p.numel() for p in backbone.parameters())
print(f"Total number of parameters: {total_params}")

Total number of parameters: 13260097


In [15]:
for name, child in backbone.named_children():
    print(name, child)

tokenizer Tokenizer(
  (conv_layers): Sequential(
    (0): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): ReLU()
      (2): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(64, 384, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): ReLU()
      (2): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    )
  )
  (flattener): Flatten(start_dim=2, end_dim=3)
)
classifier TransformerClassifier(
  (attention_pool): Linear(in_features=384, out_features=1, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0): TransformerEncoderLayer(
      (pre_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (self_attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=False)
        (attn_drop): Dropout(p=0.1, inplace=False)
        (proj): Linear(in_fe

In [ ]:
for name, child in backbone.named_children():
    for param in child.parameters():
        print(name, param.requires_grad)